# 01 — Exploratory data analysis

Load competition files from `../data/` and inspect target balance, missing values, and categoricals.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = Path("..") / "data"
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_nolabel.csv")

train.head(), test.head(), train.shape, test.shape

In [ ]:
TARGET = "Accept"

print("Train columns:", list(train.columns))
print("Test columns:", list(test.columns))
print("\nDtypes:\n", train.dtypes)
print("\nMissing (train):\n", train.isna().sum().sort_values(ascending=False))
print("\nMissing (test):\n", test.isna().sum().sort_values(ascending=False))

In [ ]:
if TARGET in train.columns:
    balance = train[TARGET].value_counts(normalize=True)
    display(balance)
    train[TARGET].value_counts().plot(kind="bar", title=f"Target: {TARGET}")
    plt.tight_layout()
    plt.show()
else:
    print(f"Expected target column {TARGET!r} not found — check competition data dictionary.")

In [ ]:
cat_cols = train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = train.select_dtypes(include="number").columns.tolist()
if TARGET in num_cols:
    num_cols.remove(TARGET)

print("Categorical:", cat_cols)
print("Numeric:", num_cols)

for col in cat_cols:
    print(f"\n{col}: {train[col].nunique()} levels")
    print(train[col].value_counts().head(10))

In [ ]:
# Leakage check: columns in train but not test (except target)
train_only = set(train.columns) - set(test.columns) - {TARGET}
test_only = set(test.columns) - set(train.columns)
print("Train-only columns (should be target or engineered):", train_only)
print("Test-only columns:", test_only)

shared = [c for c in train.columns if c in test.columns and c != TARGET]
if shared and TARGET in train.columns:
    fig, axes = plt.subplots(1, min(3, len(shared)), figsize=(4 * min(3, len(shared)), 3))
    if len(shared) == 1:
        axes = [axes]
    for ax, col in zip(axes, shared[:3]):
        if train[col].dtype == "object" or train[col].nunique() < 15:
            pd.crosstab(train[col], train[TARGET], normalize="index").plot(kind="bar", ax=ax, stacked=True)
            ax.set_title(col)
    plt.tight_layout()
    plt.show()

**Next:** `02_baseline.ipynb` — sklearn pipeline and first submission (`id`, `Accept`).